# RaV-IDP: Reconstruction-as-Validation on the Draftly corpus

An implementation of **RaV-IDP: A Reconstruction-as-Validation Framework for
Faithful Intelligent Document Processing** (Pritesh Jha, April 2026), applied to
the matter bundles in `ocr-benchmark/cases/`.

Formulas and thresholds follow the [reference
implementation](https://github.com/pritesh-2711/RaV-IDP) rather than only the
paper, because the code deviates from the prose in one place that matters (noted
in the table comparator below).

## Why this paper is the most useful of the three for Draftly

The benchmark's central problem is that **there is almost no ground truth** — 37
labelled fields on 4 documents, against a 282-page corpus. Every accuracy number
so far is confined to that sliver.

RaV-IDP's central claim is a **label-free quality signal**. Rather than asking a
model how confident it feels, it asks: *reconstruct the source from what you
extracted, and compare that reconstruction against the original document region.*
That score needs no annotation, so it can run on all 282 pages today.

The paper's own justification is worth quoting, because it is exactly the failure
the benchmark's invented-value metric was built to catch:

> Model confidence scores measure the probability the model assigns to its own
> prediction — not whether the prediction matches the document. A model that is
> systematically wrong produces confidently wrong outputs.

## The bootstrap constraint

The one idea that makes this non-circular:

> The comparator always anchors against the original document region, never
> against the extraction. If the comparator anchors against the extraction, a
> wrong extraction that reconstructs cleanly always passes. The constraint is
> enforced at the interface level: the comparator has no access to the extraction.

That is enforced here in the type system, not by convention — the comparator
receives a frozen `SourceRegion` and a reconstruction, and there is no field on
either that carries the extracted entity.

## What this run can and cannot show

**Can:** whether the fidelity score actually predicts correctness on this corpus.
The paper reports Spearman ρ = 0.800 against ground-truth table quality. The same
correlation is computed below against the labelled fields — if it holds, the
signal is usable on the 245 unlabelled pages too, which would be the single most
valuable result available right now.

**Cannot:** reproduce the GPT-4.1 fallback. The Gemini credential returns 401, so
the fallback tier here is **docTR** — a genuinely different engine from the
primary (RapidOCR), which is what the design actually requires. The paper's own
reconstructor is "re-apply OCR to the original crop to produce an independent
text reading", so using a second engine is faithful to the intent and arguably
stronger than re-running the same one.

In [ ]:
from __future__ import annotations

import gc, json, os, re, sys, time
from collections import Counter, defaultdict
from dataclasses import dataclass, field
from pathlib import Path
from typing import Any, Callable, Sequence

# Cap CPU maths threads before torch/numpy import: docTR will otherwise exhaust
# a 16 GB machine and the kernel dies rather than raising.
os.environ.setdefault("OMP_NUM_THREADS", "2")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "2")
os.environ.setdefault("MKL_NUM_THREADS", "2")

sys.path.insert(0, str(Path.cwd() if Path.cwd().name == "ocr-benchmark" else Path.cwd() / "ocr-benchmark"))

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image
from scipy.stats import spearmanr

import config, contract, normalize, render, viz

viz.use_style()
pd.set_option("display.max_rows", 200)
pd.set_option("display.width", 170)

# Their config.py default is 150; production here rasterizes at 200, so 200 is
# used to stay comparable with the rest of the benchmark.
DPI = 200
VARIANT = "original"      # RaV-IDP pre-processes REGIONS, never the full page
MAX_PAGES_PER_DOC = 2

# Paper Table 1 / their .env defaults.
THRESHOLDS = {"table": 0.75, "image": 0.70, "text": 0.85}

CACHE = config.RENDERS / "engine-cache"   # shared with doctr_trial.ipynb
CACHE.mkdir(parents=True, exist_ok=True)

print("thresholds:", THRESHOLDS)
for case in config.case_dirs():
    print(f"  {case.name:45s} {len(config.case_documents(case)):3d} documents")

## The metrics, implemented rather than imported

Their code uses `python-Levenshtein`, `scikit-image` for SSIM and `imagehash` for
pHash. None are installed here and all three are short, well-defined functions,
so they are implemented directly — consistent with the decision already taken for
CER elsewhere in this benchmark. Each is checked against a known case below.

In [ ]:
def compute_cer(hypothesis: str, reference: str) -> float:
    """Their comparators/table.py compute_cer, including the clamp at 1.0."""
    if not reference:
        return 0.0 if not hypothesis else 1.0
    return min(1.0, normalize.levenshtein(hypothesis, reference) / len(reference))


def binarize(image: Image.Image) -> np.ndarray:
    """Otsu binarization, as in their comparators/table.py binarize()."""
    gray = np.asarray(image.convert("L"))
    _, binary = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    return binary


def ssim(a: np.ndarray, b: np.ndarray, data_range: float = 255.0) -> float:
    """Structural similarity, Wang et al. 2004, gaussian-weighted.

    Equivalent to skimage.metrics.structural_similarity with default settings
    for 2-D uint8 input. Implemented here to avoid adding scikit-image.
    """
    a = a.astype(np.float64)
    b = b.astype(np.float64)
    c1, c2 = (0.01 * data_range) ** 2, (0.03 * data_range) ** 2
    kernel = (11, 11)
    sigma = 1.5
    mu_a = cv2.GaussianBlur(a, kernel, sigma)
    mu_b = cv2.GaussianBlur(b, kernel, sigma)
    mu_a2, mu_b2, mu_ab = mu_a * mu_a, mu_b * mu_b, mu_a * mu_b
    sigma_a2 = cv2.GaussianBlur(a * a, kernel, sigma) - mu_a2
    sigma_b2 = cv2.GaussianBlur(b * b, kernel, sigma) - mu_b2
    sigma_ab = cv2.GaussianBlur(a * b, kernel, sigma) - mu_ab
    numerator = (2 * mu_ab + c1) * (2 * sigma_ab + c2)
    denominator = (mu_a2 + mu_b2 + c1) * (sigma_a2 + sigma_b2 + c2)
    return float(np.mean(numerator / denominator))


def phash(image: Image.Image, hash_size: int = 8, highfreq_factor: int = 4) -> np.ndarray:
    """DCT perceptual hash, the imagehash.phash algorithm."""
    size = hash_size * highfreq_factor
    gray = np.asarray(image.convert("L").resize((size, size), Image.LANCZOS), dtype=np.float64)
    dct = cv2.dct(gray)
    low = dct[:hash_size, :hash_size]
    median = np.median(low[1:].flatten() if low.size > 1 else low)
    return (low > median).flatten()


def phash_similarity(a: np.ndarray, b: np.ndarray) -> float:
    """Their comparators/image.py: 1 - hamming/64."""
    return 1.0 - float(np.count_nonzero(a != b)) / a.size


def sharpness(image: Image.Image) -> float:
    """Variance of the Laplacian, their reconstructors/image.py signal."""
    return float(cv2.Laplacian(np.asarray(image.convert("L")), cv2.CV_64F).var())


# --- checks -----------------------------------------------------------------
assert compute_cer("abc", "abc") == 0.0
assert compute_cer("abX", "abc") == 1 / 3
assert compute_cer("anything", "") == 1.0
assert compute_cer("", "") == 0.0

noise = np.random.default_rng(0).integers(0, 255, (128, 128)).astype(np.uint8)
assert abs(ssim(noise, noise) - 1.0) < 1e-6, "SSIM of an image with itself must be 1"
shifted = np.roll(noise, 40, axis=0)
print(f"SSIM  self={ssim(noise, noise):.4f}  vs shifted={ssim(noise, shifted):.4f}")

img = Image.fromarray(noise)
assert phash_similarity(phash(img), phash(img)) == 1.0
print(f"pHash self={phash_similarity(phash(img), phash(img)):.3f}  "
      f"vs shifted={phash_similarity(phash(img), phash(Image.fromarray(shifted))):.3f}")

## The bootstrap constraint, enforced by the types

`SourceRegion` is frozen and holds the immutable crop taken from the page. It has
no field for the extraction, so a comparator physically cannot anchor against
what was extracted. The paper is explicit that this must be an interface-level
guarantee rather than a convention, because a comparator that peeks at the
extraction will pass a consistently wrong extractor every time.

In [ ]:
@dataclass(frozen=True)
class SourceRegion:
    """The immutable reference. Every comparator receives this and nothing else."""
    region_id: str
    doc_id: str
    page_no: int
    bbox: tuple[float, float, float, float]
    entity_type: str                 # table | image | text
    crop: Image.Image                # the unmodified crop from the original page


@dataclass
class Entity:
    """What an extractor produced. Deliberately NOT visible to any comparator."""
    region_id: str
    entity_type: str
    text: str = ""
    rows: int = 0
    cols: int = 0
    cells: list[str] = field(default_factory=list)
    extractor: str = "primary"
    seconds: float = 0.0


@dataclass
class Reconstruction:
    """The entity rendered back into something comparable with the source."""
    region_id: str
    reocr_text: str = ""                       # independent re-read
    rendered: Image.Image | None = None        # rasterized table grid
    structural_signature: dict[str, Any] = field(default_factory=dict)
    phash_bits: np.ndarray | None = None
    sharpness_crop: float = 0.0
    sharpness_original: float = 0.0
    caption_found: bool = False


@dataclass
class FidelityResult:
    region_id: str
    entity_type: str
    fidelity_score: float
    passed_threshold: bool
    threshold_used: float
    component_scores: dict[str, float]
    extractor_name: str = "primary"


print("SourceRegion fields:", [f for f in SourceRegion.__dataclass_fields__])
assert "entity" not in SourceRegion.__dataclass_fields__
assert "extraction" not in SourceRegion.__dataclass_fields__
print("bootstrap constraint holds: the reference carries no extraction")

## Components 1–4: quality, layout, region pre-processing, routing

Their architectural note, which is easy to get backwards:

> Layout detection runs before classification and pre-processing, because the
> layout detector requires the raw document. Pre-processing is applied to
> individual regions after they are detected, not to the full page before
> detection.

That is why `VARIANT = "original"` above — the page is deliberately *not*
preprocessed. Regions are cleaned individually after detection.

In [ ]:
_RAPID = None
_DOCTR = None

def rapid():
    global _RAPID
    if _RAPID is None:
        from rapidocr import RapidOCR
        _RAPID = RapidOCR()
    return _RAPID

def doctr():
    global _DOCTR
    if _DOCTR is None:
        from doctr.models import ocr_predictor
        import torch
        torch.set_num_threads(2); torch.set_grad_enabled(False)
        _DOCTR = ocr_predictor(pretrained=True)
    return _DOCTR


def quality_classifier(page: render.RenderedPage) -> dict[str, Any]:
    """Component 1. Their version is skew/sharpness based, kept simple as in the repo."""
    skew = render.estimate_skew(page.image)
    sharp = sharpness(page.image)
    label = "clean_scan"
    if abs(skew) > 2.0:
        label = "skewed"
    if sharp < 30.0:
        label = "degraded"
    return {"quality": label, "skew": round(skew, 2), "sharpness": round(sharp, 1)}


def layout_detector(page: render.RenderedPage, doc_id: str) -> list[dict[str, Any]]:
    """Component 2. Detect regions on the RAW page. RapidOCR boxes, grouped into blocks."""
    words = ocr_boxes(page, doc_id, "rapidocr")
    if not words:
        return []
    # Group word boxes into coarse regions by vertical proximity, so a region is
    # a paragraph/table-ish block rather than a single word.
    words = sorted(words, key=lambda w: (w["bbox"][1], w["bbox"][0]))
    regions: list[list[dict]] = []
    for word in words:
        if regions and word["bbox"][1] - regions[-1][-1]["bbox"][3] < 0.02:
            regions[-1].append(word)
        else:
            regions.append([word])
    out = []
    for i, group in enumerate(regions):
        xs0 = min(w["bbox"][0] for w in group); ys0 = min(w["bbox"][1] for w in group)
        xs1 = max(w["bbox"][2] for w in group); ys1 = max(w["bbox"][3] for w in group)
        out.append({"idx": i, "bbox": (xs0, ys0, xs1, ys1), "words": group})
    return out


def region_preprocessor(crop: Image.Image) -> Image.Image:
    """Component 3. Applied per REGION after detection, never to the page before it."""
    array = np.asarray(crop.convert("L"))
    if array.size == 0:
        return crop
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    return Image.fromarray(clahe.apply(array))


def entity_router(region: dict[str, Any]) -> str:
    """Component 4. table | image | text.

    A region with several well-separated columns on multiple lines is treated as
    a table; a region with almost no recognised text is an image/diagram; the
    rest is text. This is a heuristic router standing in for their layout model,
    and it is the weakest link in this reimplementation - flagged rather than
    hidden.
    """
    words = region["words"]
    if len(words) < 3:
        return "image"
    ys = sorted({round(w["bbox"][1], 2) for w in words})
    xs = sorted({round(w["bbox"][0], 2) for w in words})
    if len(ys) >= 3 and len(xs) >= 3:
        gaps = [xs[i + 1] - xs[i] for i in range(len(xs) - 1)]
        if gaps and max(gaps) > 0.08:
            return "table"
    return "text"


print("components 1-4 ready")

In [ ]:
def _cache_path(engine: str, doc_id: str, page_no: int) -> Path:
    safe = "".join(c if c.isalnum() or c in "-_." else "-" for c in doc_id)
    return CACHE / engine / safe / f"p{page_no:04d}@{DPI}-{VARIANT}.json"


def ocr_boxes(page: render.RenderedPage, doc_id: str, engine: str) -> list[dict[str, Any]]:
    """Word boxes from either engine, cached. Shares doctr_trial.ipynb's cache."""
    cached = _cache_path(engine, doc_id, page.page_no)
    if cached.is_file():
        return json.loads(cached.read_text(encoding="utf-8"))["words"]

    began = time.time()
    words: list[dict[str, Any]] = []
    w, h = page.size
    if engine == "rapidocr":
        result = rapid()(np.asarray(page.image.convert("RGB")))
        boxes = getattr(result, "boxes", None)
        txts = getattr(result, "txts", None) or []
        scores = getattr(result, "scores", None) or []
        if boxes is not None:
            for i, poly in enumerate(boxes):
                words.append({"text": str(txts[i]) if i < len(txts) else "",
                              "bbox": list(contract.bbox_from_polygon(poly, w, h)),
                              "confidence": float(scores[i]) if i < len(scores) else 0.0,
                              "block": 0, "line": i})
    else:
        export = doctr()([np.asarray(page.image.convert("RGB"))]).export()
        for b, block in enumerate(export["pages"][0]["blocks"]):
            for l, line in enumerate(block["lines"]):
                for word in line["words"]:
                    (x0, y0), (x1, y1) = word["geometry"]
                    words.append({"text": word["value"],
                                  "bbox": list(contract.clamp_bbox((x0, y0, x1, y1))),
                                  "confidence": float(word["confidence"]),
                                  "block": b, "line": l})

    payload = {"engine": engine, "words": words, "seconds": round(time.time() - began, 2),
               "blocks": len({w_["block"] for w_ in words}) or 1, "lines": len(words)}
    cached.parent.mkdir(parents=True, exist_ok=True)
    cached.write_text(json.dumps(payload, ensure_ascii=False), encoding="utf-8")
    return words


def words_in(words: list[dict[str, Any]], bbox: Sequence[float]) -> list[dict[str, Any]]:
    """Words whose centre falls inside a region box."""
    x0, y0, x1, y1 = bbox
    out = []
    for word in words:
        cx = (word["bbox"][0] + word["bbox"][2]) / 2
        cy = (word["bbox"][1] + word["bbox"][3]) / 2
        if x0 <= cx <= x1 and y0 <= cy <= y1:
            out.append(word)
    return out


def lines_from(words: list[dict[str, Any]], tol: float = 0.012) -> list[str]:
    """Reading-order lines, used as the structural signature for tables."""
    if not words:
        return []
    rows: list[list[dict]] = []
    for word in sorted(words, key=lambda w: (w["bbox"][1], w["bbox"][0])):
        centre = (word["bbox"][1] + word["bbox"][3]) / 2
        placed = False
        for row in rows:
            ref = (row[0]["bbox"][1] + row[0]["bbox"][3]) / 2
            if abs(centre - ref) <= tol:
                row.append(word); placed = True; break
        if not placed:
            rows.append([word])
    return [" ".join(w["text"] for w in sorted(r, key=lambda w: w["bbox"][0])).strip()
            for r in rows if any(w["text"].strip() for w in r)]


print("ocr helpers ready")

## Components 5–6: extractors and reconstructors

The reconstructor is the heart of the design. Per the paper:

> **Text reconstructor:** re-applies OCR to the original crop to produce an
> independent text reading.
> **Table reconstructor:** renders the DataFrame as an HTML table, rasterizes to
> a pixel image matching the source crop.
> **Image reconstructor:** computes the perceptual hash, the Laplacian sharpness,
> and a caption adjacency check.

Here the independent re-read is **docTR** rather than a second pass of RapidOCR.
Two different engines disagreeing is a stronger independence guarantee than one
engine run twice, which would be deterministic and therefore useless.

In [ ]:
def extract_entity(region: SourceRegion, page_words: list[dict[str, Any]],
                   extractor: str = "primary") -> Entity:
    """Component 5. `primary` = RapidOCR page read; `fallback` = docTR on the crop."""
    began = time.time()
    if extractor == "primary":
        words = words_in(page_words, region.bbox)
    else:
        # Fallback re-extracts from the region itself with the stronger engine.
        cleaned = region_preprocessor(region.crop)
        tmp = render.RenderedPage(region.page_no, cleaned, DPI, VARIANT,
                                  contract.IDENTITY, False)
        export = doctr()([np.asarray(tmp.image.convert("RGB"))]).export()
        words = []
        for block in export["pages"][0]["blocks"]:
            for line in block["lines"]:
                for word in line["words"]:
                    (x0, y0), (x1, y1) = word["geometry"]
                    words.append({"text": word["value"], "bbox": [x0, y0, x1, y1],
                                  "confidence": float(word["confidence"])})
    lines = lines_from(words)
    cols = max((len(l.split()) for l in lines), default=0)
    return Entity(region_id=region.region_id, entity_type=region.entity_type,
                  text="\n".join(lines), rows=len(lines), cols=cols, cells=lines,
                  extractor=extractor, seconds=round(time.time() - began, 2))


def reconstruct(region: SourceRegion, entity: Entity) -> Reconstruction:
    """Component 6. Render the entity back to something comparable with the source.

    Note the asymmetry that makes this work: the RE-READ is taken from
    `region.crop` (the document), never from `entity`. The entity is only used
    for the parts that are genuinely a rendering of the extraction, such as the
    rasterized table grid.
    """
    rec = Reconstruction(region_id=region.region_id)

    if region.entity_type in ("text", "table"):
        # Independent re-read of the ORIGINAL crop with the other engine.
        cleaned = region_preprocessor(region.crop)
        export = doctr()([np.asarray(cleaned.convert("RGB"))]).export()
        reread = []
        for block in export["pages"][0]["blocks"]:
            for line in block["lines"]:
                reread.append(" ".join(w["value"] for w in line["words"]))
        rec.reocr_text = "\n".join(reread)

    if region.entity_type == "table":
        rec.rendered = render_table_image(entity, region.crop.size)
        rec.structural_signature = {"row_count": entity.rows, "col_count": entity.cols,
                                    "cells": entity.cells}

    if region.entity_type == "image":
        rec.phash_bits = phash(region.crop)
        rec.sharpness_crop = sharpness(region.crop)
        rec.sharpness_original = max(rec.sharpness_crop, 1e-9)
        rec.caption_found = bool(entity.text.strip())
    return rec


def render_table_image(entity: Entity, size: tuple[int, int]) -> Image.Image:
    """Rasterize the extracted grid, standing in for their HTML->image step."""
    width, height = max(size[0], 32), max(size[1], 32)
    canvas = Image.new("L", (width, height), 255)
    from PIL import ImageDraw
    draw = ImageDraw.Draw(canvas)
    rows = max(entity.rows, 1)
    row_h = height / rows
    for r, line in enumerate(entity.cells[:rows]):
        y = r * row_h
        draw.line([(0, y), (width, y)], fill=0, width=1)
        draw.text((4, y + 2), line[:120], fill=0)
    return canvas


print("extractors and reconstructors ready")

## Component 7: comparators, fidelity, and the fallback loop

Formulas exactly as in their code:

```text
f_text   = max(0, 1 - CER(extracted, re-OCR of the original crop))
f_table  = 0.4 * SSIM_binarized + 0.6 * f_struct
f_struct = 0.2 * row_col_match + 0.8 * (1 - CER_cells)
f_image  = 0.6 * pHash_sim + 0.3 * sharpness_ratio + 0.1 * caption_score
```

One deliberate deviation from the paper, taken from their code and their comment
explaining it: the paper's `f_struct` includes a `0.3 * (1 - CER_headers)` term,
but OCR on a table image yields one line per row rather than one per column, so
header CER is self-referential. The header weight is absorbed into the cell term,
keeping the weights summing to 1.

In [ ]:
def _has_visual_content(crop: Image.Image, min_dark_ratio: float = 0.04) -> bool:
    gray = np.asarray(crop.convert("L"))
    if gray.size == 0:
        return False
    _, binary = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)
    return float(binary.sum()) / (255.0 * binary.size) > min_dark_ratio


def _soft_count_match(predicted: int, reference: int, crop: Image.Image) -> float:
    """Their _soft_count_match, including the empty-vs-visual-content guard."""
    if predicted == 0 and reference == 0:
        return 0.0 if _has_visual_content(crop) else 1.0
    if predicted == 0 or reference == 0:
        return 0.0
    return min(predicted, reference) / max(predicted, reference)


def compare_text(reconstruction: Reconstruction, extracted_text: str,
                 region: SourceRegion, threshold: float) -> FidelityResult:
    reference = reconstruction.reocr_text
    if not reference:
        score = 1.0 if not extracted_text else 0.0
        cer = 0.0 if score == 1.0 else 1.0
    else:
        cer = compute_cer(extracted_text, reference)
        score = max(0.0, 1.0 - cer)
    return FidelityResult(region.region_id, "text", round(score, 4),
                          score >= threshold, threshold, {"cer": round(cer, 4)})


def compare_table(reconstruction: Reconstruction, region: SourceRegion,
                  threshold: float, skip_visual: bool = False) -> FidelityResult:
    signature = reconstruction.structural_signature

    visual = None
    if not skip_visual and reconstruction.rendered is not None:
        rendered = binarize(reconstruction.rendered)
        original = binarize(region.crop)
        original = cv2.resize(original, (rendered.shape[1], rendered.shape[0]))
        visual = max(0.0, min(1.0, ssim(rendered, original)))

    ocr_lines = [l for l in reconstruction.reocr_text.splitlines() if l.strip()]
    ocr_cols = max((len(re.split(r"\t|\s{2,}", l.strip())) for l in ocr_lines), default=0)

    row_match = _soft_count_match(signature.get("row_count", 0), len(ocr_lines), region.crop)
    col_match = _soft_count_match(signature.get("col_count", 0), ocr_cols, region.crop)
    row_col_match = (row_match + col_match) / 2.0
    cer_cells = compute_cer(" ".join(signature.get("cells", [])), " ".join(ocr_lines))

    f_struct = max(0.0, min(1.0, 0.2 * row_col_match + 0.8 * (1.0 - cer_cells)))
    score = f_struct if visual is None else max(0.0, min(1.0, 0.4 * visual + 0.6 * f_struct))

    components = {"f_struct": round(f_struct, 4), "row_match": round(row_match, 4),
                  "col_match": round(col_match, 4), "cer_cells": round(cer_cells, 4)}
    if visual is not None:
        components["ssim"] = round(visual, 4)
    return FidelityResult(region.region_id, "table", round(score, 4),
                          score >= threshold, threshold, components)


def compare_image(reconstruction: Reconstruction, region: SourceRegion,
                  threshold: float) -> FidelityResult:
    original_bits = phash(region.crop)
    similarity = 1.0 if reconstruction.phash_bits is None else max(
        0.0, min(1.0, phash_similarity(reconstruction.phash_bits, original_bits)))
    ratio = (min(reconstruction.sharpness_crop / reconstruction.sharpness_original, 1.0)
             if reconstruction.sharpness_original else 0.0)
    caption = 1.0 if reconstruction.caption_found else 0.8
    score = max(0.0, min(1.0, 0.6 * similarity + 0.3 * ratio + 0.1 * caption))
    return FidelityResult(region.region_id, "image", round(score, 4),
                          score >= threshold, threshold,
                          {"phash_similarity": round(similarity, 4),
                           "sharpness_ratio": round(ratio, 4),
                           "caption_score": round(caption, 4)})


COMPARATORS: dict[str, Callable[..., FidelityResult]] = {
    "text": lambda rec, reg, thr, ent: compare_text(rec, ent.text, reg, thr),
    "table": lambda rec, reg, thr, ent: compare_table(rec, reg, thr),
    "image": lambda rec, reg, thr, ent: compare_image(rec, reg, thr),
}

print("comparators ready")

In [ ]:
def validate_region(region: SourceRegion, page_words: list[dict[str, Any]],
                    enable_fallback: bool = True, max_attempts: int = 2
                    ) -> dict[str, Any]:
    """Extract -> reconstruct -> compare -> fall back if fidelity < threshold.

    The loop is the whole point: the paper's gate-only ablation collapses
    (0.4224 -> 0.1408 ANLS), showing the value is in ROUTING low-fidelity regions
    to a stronger extractor, not in discarding them.
    """
    threshold = THRESHOLDS[region.entity_type]
    attempts: list[FidelityResult] = []
    entity = extract_entity(region, page_words, "primary")

    for attempt in range(max_attempts):
        reconstruction = reconstruct(region, entity)
        result = COMPARATORS[region.entity_type](reconstruction, region, threshold, entity)
        result.extractor_name = entity.extractor
        attempts.append(result)
        if result.passed_threshold or not enable_fallback or attempt == max_attempts - 1:
            break
        entity = extract_entity(region, page_words, "fallback")

    final = attempts[-1]
    return {
        "region_id": region.region_id,
        "doc_id": region.doc_id,
        "page": region.page_no,
        "entity_type": region.entity_type,
        "fidelity": final.fidelity_score,
        "passed": final.passed_threshold,
        "threshold": final.threshold_used,
        "extractor": final.extractor_name,
        "attempts": len(attempts),
        "recovered": len(attempts) > 1 and final.passed_threshold,
        "components": final.component_scores,
        "text": entity.text,
        "first_fidelity": attempts[0].fidelity_score,
    }

print("validation loop ready")

## Run

Every region is extracted, reconstructed and scored. docTR is ~50 s per region
read on CPU, so this is capped tightly and cached; the cache is shared with
`doctr_trial.ipynb`, so pages already read there are free.

In [ ]:
MATTER = "platform-case-001"
MAX_REGIONS_PER_PAGE = 6      # largest regions only; docTR cost is per region
MIN_FREE_GB = 1.2

def free_ram_gb() -> float:
    try:
        import ctypes
        class S(ctypes.Structure):
            _fields_ = [("dwLength", ctypes.c_ulong), ("dwMemoryLoad", ctypes.c_ulong),
                        ("ullTotalPhys", ctypes.c_ulonglong), ("ullAvailPhys", ctypes.c_ulonglong),
                        ("ullTotalPageFile", ctypes.c_ulonglong), ("ullAvailPageFile", ctypes.c_ulonglong),
                        ("ullTotalVirtual", ctypes.c_ulonglong), ("ullAvailVirtual", ctypes.c_ulonglong),
                        ("ullAvailExtendedVirtual", ctypes.c_ulonglong)]
        s = S(); s.dwLength = ctypes.sizeof(S)
        ctypes.windll.kernel32.GlobalMemoryStatusEx(ctypes.byref(s))
        return s.ullAvailPhys / 1024 ** 3
    except Exception:
        return float("inf")

cases = [c for c in config.case_dirs() if MATTER is None or c.name == MATTER]
documents = [d for c in cases for d in config.case_documents(c)]
print(f"{len(documents)} documents | free RAM {free_ram_gb():.1f} GB\n")

rows: list[dict[str, Any]] = []
stopped = False

for doc in documents:
    if stopped:
        break
    doc_id = f"{doc.parent.name}/{doc.name}"
    for page in render.render_doc(doc, dpi=DPI, variant=VARIANT, page_limit=MAX_PAGES_PER_DOC):
        if free_ram_gb() < MIN_FREE_GB:
            print(f"\nstopping early: {free_ram_gb():.1f} GB free. Progress is cached.")
            stopped = True
            break
        quality = quality_classifier(page)
        page_words = ocr_boxes(page, doc_id, "rapidocr")
        detected = layout_detector(page, doc_id)
        detected.sort(key=lambda r: -( (r["bbox"][2]-r["bbox"][0]) * (r["bbox"][3]-r["bbox"][1]) ))
        detected = detected[:MAX_REGIONS_PER_PAGE]

        w, h = page.size
        for region_spec in detected:
            etype = entity_router(region_spec)
            px = contract.bbox_to_pixels(region_spec["bbox"], w, h)
            if px[2] - px[0] < 8 or px[3] - px[1] < 8:
                continue
            source = SourceRegion(
                region_id=f"{doc.stem[:18]}-p{page.page_no}-r{region_spec['idx']}",
                doc_id=doc_id, page_no=page.page_no,
                bbox=tuple(region_spec["bbox"]), entity_type=etype,
                crop=page.image.crop(px),
            )
            record = validate_region(source, page_words, enable_fallback=True)
            record["quality"] = quality["quality"]
            rows.append(record)
        page.image.close()
        gc.collect()
    print(f"  {doc.name[:50]:52s} regions so far: {len(rows)}  ({free_ram_gb():.1f} GB free)")

rav = pd.DataFrame(rows)
print(f"\n{len(rav)} regions validated")
if len(rav):
    print(rav.groupby("entity_type")[["fidelity", "passed", "attempts"]].mean().round(3).to_string())

In [ ]:
if len(rav):
    total = len(rav)
    passed = int(rav["passed"].sum())
    fell_back = int((rav["attempts"] > 1).sum())
    recovered = int(rav["recovered"].sum())
    print(f"regions               {total}")
    print(f"passed the gate       {passed} ({passed/total:.1%})")
    print(f"routed to fallback    {fell_back} ({fell_back/total:.1%})")
    print(f"recovered by fallback {recovered}"
          f"{f' ({recovered/fell_back:.1%} of those routed)' if fell_back else ''}")
    print(f"\npaper's comparable figure: 38.1% of failed table extractions recovered")
    display(rav[["region_id", "entity_type", "first_fidelity", "fidelity",
                 "threshold", "passed", "attempts", "extractor"]].head(25))

## The experiment that matters: does fidelity predict correctness?

Everything above is label-free. The question is whether it is *worth* anything —
a fidelity score that does not track real correctness is an expensive random
number.

The paper reports Spearman ρ = 0.800 between fidelity and ground-truth table
quality (ρ = 0.877 on native PDFs). The same correlation is computed here against
the only ground truth available: whether a labelled field's true value appears
verbatim in the region's extracted text.

If this correlates on our corpus, the score can be trusted on the 245 unlabelled
pages — which would be the most useful thing this benchmark has produced.

In [ ]:
expected: dict[str, dict[str, Any]] = {}
for case in config.case_dirs():
    path = config.expected_fields_path(case)
    if path.is_file():
        for name, payload in json.loads(path.read_text(encoding="utf-8")).items():
            expected[f"{case.name}/{name}"] = payload

pairs = []
for _, row in rav.iterrows() if len(rav) else []:
    truth = expected.get(row["doc_id"], {}).get("fields", {})
    if not truth:
        continue
    # Correctness proxy: share of this document's true values present verbatim in
    # the region text. A faithful extraction of a region containing a field
    # should contain that field's value.
    present = [normalize.value_in_text(str(v), row["text"]) for v in truth.values()]
    if not any(present):
        continue   # region contains none of the labelled fields; nothing to correlate
    pairs.append({"fidelity": row["fidelity"], "correct": float(np.mean(present)),
                  "entity_type": row["entity_type"], "region_id": row["region_id"]})

corr = pd.DataFrame(pairs)
if len(corr) < 3:
    print(f"only {len(corr)} regions overlap a labelled field — too few to correlate.")
    print("Raise MAX_PAGES_PER_DOC / MAX_REGIONS_PER_PAGE, or label a second matter.")
else:
    rho, p = spearmanr(corr["fidelity"], corr["correct"])
    print(f"n = {len(corr)} regions")
    print(f"Spearman rho = {rho:.3f}  (p = {p:.3g})")
    print(f"paper: rho = 0.800 on DocLayNet tables, 0.877 on native PDFs")
    if p > 0.05:
        print("\nNot significant at this sample size. That is a statement about n,")
        print("not evidence that the signal is worthless.")

In [ ]:
if len(rav):
    fig, axes = plt.subplots(1, 3, figsize=(16, 4.2))

    ax = axes[0]
    for i, etype in enumerate(sorted(rav["entity_type"].unique())):
        vals = rav[rav.entity_type == etype]["fidelity"]
        ax.hist(vals, bins=12, alpha=0.75, color=viz.SERIES[i % 4], label=etype)
        ax.axvline(THRESHOLDS[etype], color=viz.MUTED, linestyle="--", linewidth=1)
    ax.set_xlabel("fidelity score")
    ax.set_ylabel("regions")
    ax.set_title("Fidelity distribution (dashed = acceptance threshold)")
    ax.legend()

    ax = axes[1]
    types = sorted(rav["entity_type"].unique())
    x = np.arange(len(types))
    pass_rate = [rav[rav.entity_type == t]["passed"].mean() for t in types]
    fb_rate = [(rav[rav.entity_type == t]["attempts"] > 1).mean() for t in types]
    ax.bar(x - 0.2, pass_rate, 0.36, color=viz.STATUS["correct"], label="passed gate")
    ax.bar(x + 0.2, fb_rate, 0.36, color=viz.STATUS["missing"], label="routed to fallback")
    ax.set_xticks(x, types)
    ax.set_ylabel("share of regions")
    ax.set_title("Gate outcome by entity type")
    ax.legend()

    ax = axes[2]
    if len(corr) >= 3:
        ax.scatter(corr["fidelity"], corr["correct"], s=90, color=viz.SEQ_HUE,
                   alpha=0.85, edgecolor=viz.SURFACE, linewidth=2)
        ax.set_xlabel("fidelity (label-free)")
        ax.set_ylabel("share of true values present")
        ax.set_title(f"Does fidelity track correctness?  rho = {spearmanr(corr['fidelity'], corr['correct'])[0]:.2f}")
    else:
        ax.text(0.5, 0.5, "not enough labelled overlap\nto correlate", ha="center",
                va="center", color=viz.MUTED, transform=ax.transAxes)
        ax.set_xticks([]); ax.set_yticks([]); ax.grid(False)
        ax.set_title("Does fidelity track correctness?")

    plt.tight_layout(); plt.show()

## Ablation: gate-only versus gate-plus-fallback

The paper's sharpest result. A fidelity gate that merely *excludes* low-scoring
regions collapses performance (0.4224 → 0.1408 ANLS), because throwing away hard
regions throws away the answers with them. The value is in **routing** them to a
stronger extractor.

Measured here as retained text volume and retained labelled values.

In [ ]:
if len(rav):
    def retained(df: pd.DataFrame, mode: str) -> dict[str, Any]:
        if mode == "no_gate":
            keep = df
        elif mode == "gate_only":
            keep = df[df["first_fidelity"] >= df["threshold"]]   # discard low fidelity
        else:
            keep = df[df["passed"]]                              # gate + fallback
        chars = int(keep["text"].str.replace(r"\s+", "", regex=True).str.len().sum())
        values = 0
        for _, row in keep.iterrows():
            truth = expected.get(row["doc_id"], {}).get("fields", {})
            values += sum(1 for v in truth.values()
                          if normalize.value_in_text(str(v), row["text"]))
        return {"configuration": mode, "regions_kept": len(keep),
                "chars_kept": chars, "labelled_values_kept": values}

    ab = pd.DataFrame([retained(rav, m) for m in ["no_gate", "gate_only", "gate_and_fallback"]])
    display(ab)

    fig, ax = plt.subplots(figsize=(9, 3.4))
    x = np.arange(len(ab))
    base = max(ab["chars_kept"].max(), 1)
    ax.bar(x, ab["chars_kept"] / base, 0.6,
           color=[viz.SEQ_STEPS[2], viz.STATUS["wrong"], viz.STATUS["correct"]])
    for i, v in enumerate(ab["chars_kept"] / base):
        ax.text(i, v + 0.02, f"{v:.0%}", ha="center", color=viz.INK_2, fontsize=9)
    ax.set_xticks(x, ab["configuration"])
    ax.set_ylabel("text retained (share of ungated)")
    ax.set_title("Gating without fallback discards content; routing keeps it")
    plt.tight_layout(); plt.show()

## What to take from this

**The idea worth adopting regardless of the numbers.** Reconstruction-as-
validation gives a quality signal with **no labels**, and this project's binding
constraint is that it has almost none — 37 labelled fields against 282 pages.
Every other measurement in the benchmark is stuck behind annotation work.
Fidelity is not.

**It composes with what is already built.** The benchmark already computes an
invented-value rate — is the extracted value verbatim in the page text — which is
the same instinct in a weaker form: a single check against one reading. RaV
generalises it to a scored comparison against an *independent* re-read, per
region, with a threshold and a defined action when it fails. And the MADP
notebook's Validator already has the escalation queue that a failed fidelity
check should feed.

**The bootstrap constraint is the load-bearing part.** It is tempting to compare
the reconstruction against the extraction — it is easier and the numbers look
better. It is also worthless: a consistently wrong extractor reconstructs its own
mistake perfectly and passes every time. Enforcing it in the type signature
rather than in a code review comment is the right call.

### Honest limits of this run

- **The entity router is the weak link.** The paper uses a trained layout model;
  this uses a geometry heuristic over OCR boxes. Table/image/text assignment is
  therefore noisy, and the table and image fidelity numbers inherit that noise.
  The text path is the trustworthy one.
- **The independent re-read is Latin-only.** docTR cannot read Sinhala, so on
  Sinhala regions both engines produce noise and CER between two noisy readings
  is meaningless. Fidelity on those regions should be read as "unmeasurable",
  not as "low quality".
- **Sample size.** The correlation above is computed on however many regions
  happened to overlap a labelled field. If that count is small, the ρ is
  indicative at best — the paper's own figure rests on 500 tables.
- **No GPT-4.1 tier.** The paper's fallback is a stronger *vision-language*
  model; ours is a stronger OCR engine. That is a weaker escalation than the
  design intends, and it caps how much the fallback path can recover.

### Recommended next step

Fold the text fidelity score into the main benchmark as a per-field provenance
signal. It needs no annotation, it runs on the whole corpus, and it answers the
fourth benchmark question — *can the system show where every value came from* —
with a number rather than an assertion.